#**Caso de estudio**
En este documento se desarrollará y analizará un modelo Naive Bayes para la detección de posibles casos de lavado de activos en una entidad financiera, utilizando variables socioeconómicas y financieras asociadas al comportamiento transaccional de los clientes.

De acuerdo con lo anterior, las variables consideradas en el modelo son:

Edad: Es la edad que posee el cliente al momento del análisis, variable que puede estar relacionada con su historial financiero y perfil de riesgo.

Ingresos (USD): Son los ingresos reportados por el cliente como resultado de su actividad económica, los cuales permiten evaluar la coherencia entre capacidad financiera y volumen de operaciones.

Gastos (USD): Corresponden a los egresos mensuales del cliente, incluyendo obligaciones financieras y gastos personales, que permiten analizar la proporción entre ingresos y movimientos financieros.

Número de tarjetas de crédito: Indica la cantidad de productos financieros activos que posee el cliente, lo cual puede reflejar su nivel de bancarización.

Monto transado en tarjetas (USD): Representa el volumen total de dinero movilizado a través de tarjetas de crédito, siendo una variable clave para identificar patrones inusuales de transacción.

Porcentaje de crecimiento patrimonial: Mide la variación porcentual del patrimonio del cliente en un período determinado, lo cual puede evidenciar incrementos atípicos no justificados.

La variable de decisión corresponde a “lavado_activos”, la cual clasifica a los clientes en dos categorías: aquellos que presentan indicios de posible lavado de activos y aquellos que no presentan riesgo.



0. Carga de las librerías de trabajo


In [1]:
import numpy as np #Librería numérica por excelencia
import pandas as pd #Librería para la comunicación con archivos de excel o base de datos

#Librerías específicas
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

1. Se cargan los datos de trabajo de la base de datos BaseAAA

In [4]:
nxl='/content/2. LavadoActivos.xlsx'
XDB=pd.read_excel(nxl,sheet_name=0)
XDB.head(100) #Es para mostrar la base de datos cargada

#Seleccionamos variables de trabajo
XD = XDB[['edad',
          'ingresos_usd',
          'gastos_usd',
          'num_tarjetas_credito',
          'monto_transado_tarjetas_usd',
          'porcentaje_crecimiento_patrimonio']] #Variables de trabajo
XD.head(10)

#Variable de decisión
yd = XDB[['lavado_activos']]
yd.head()



,lavado_activos
0,1
1,1
2,0
3,0
4,0


In [5]:
# Modelo Naive Bayes
mnb = GaussianNB()
mnb.fit(XD, yd) #Ajustar variables entrada - salida

#Mostramos las medias de las variables
u = mnb.theta_
sigma = mnb.var_
sigma = np.sqrt(sigma)

print("'edad','ingresos_usd','gastos_usd','num_tarjetas_credito','monto_transado_tarjetas_usd','porcentaje_crecimiento_patrimonio'")
print(u)
print("Las desviaciones son: ")
print(sigma)

'edad','ingresos_usd','gastos_usd','num_tarjetas_credito','monto_transado_tarjetas_usd','porcentaje_crecimiento_patrimonio'
[[4.54696356e+01 6.69073255e+03 3.60174970e+03 1.95456590e+00
  3.09046404e+03 4.46871435e+01]
 [4.42998921e+01 7.02623043e+04 3.87072844e+04 2.01618123e+00
  3.38356197e+04 4.48208630e+01]]
Las desviaciones son: 
[[1.47105594e+01 6.20696435e+03 3.21901567e+03 2.53514671e+00
  2.45685863e+03 1.41731672e+01]
 [1.50523273e+01 1.13412265e+05 6.59025571e+04 2.55740732e+00
  6.29913911e+04 1.54786058e+01]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


**Análisis de resultados**

De los resultados arrojados por el modelo, podemos observar que la categoría correspondiente a los clientes asociados a lavado de activos presenta ingresos superiores en comparación con la categoría de clientes no vinculados a lavado, lo que indica claramente la predominancia de esta variable dentro del proceso de clasificación del riesgo.

Es importante mencionar que la edad promedio entre ambas categorías es relativamente similar, aunque se evidencian ligeras diferencias en el perfil demográfico. Sin embargo, la principal distinción no radica en la edad sino en el comportamiento financiero.

Podemos decir igualmente que los clientes clasificados como posibles casos de lavado presentan gastos superiores a los de la categoría sin riesgo; adicionalmente, registran montos transados en tarjetas considerablemente mayores. Esto constituye un indicador relevante del volumen de operaciones financieras y sugiere que este grupo maneja flujos monetarios más altos, lo cual puede representar un patrón atípico o de mayor exposición al riesgo de lavado de activos.

3. Se procede con la evaluación del modelo. Para la evalucaión de este tipo de modelos se utiliza la matriz de confusión


In [6]:
ydp=mnb.predict(XD) #Esto es lo que el modelo aprende - ydp de pronóstico
cm=confusion_matrix(yd,ydp)
print(cm)

#Se determinan las metricas de la matriz de confusión
VN=cm[0,0]; FP=cm[0,1]; FN=cm[1,0]; VP=cm[1,1];TDatos=len(XDB)

#1. Exactitud: funcionamiento general del modelo
Exn=(VP+VN)/TDatos
print("Exactitud: ",Exn)

#2. Tasa Error: %Fallos del modelo
TEr=(FP+FN)/TDatos
print("Tasa de error: ",TEr)

#3. Sensibilidad: como se comportó frente a los positivos solamente
Se=VP/(VP+FN)
print("sensibilidad: ", Se)

#4. Especificidad: como se comporta pronosticando negativos
Es=VN/(VN+FP)
print("Especificidad: ", Es)

#5. Precisión: es una versión de como se comporta el modelo frente a los positivos solamente
Pr=VP/(VP+FP)
print("Precisión: ",Pr)

#6. Predicción negativa: como se comporta el modelo frente a los negativos solamente
PrN=VN/(VN+FN)
print("Predicción negativa: ", PrN)

[[2179   44]
 [ 145  782]]
Exactitud:  0.94
Tasa de error:  0.06
sensibilidad:  0.8435814455231931
Especificidad:  0.9802069275753487
Precisión:  0.9467312348668281
Predicción negativa:  0.9376075731497419


**Análisis de resultados**

De acuerdo con los resultados, podemos observar de manera general que el modelo alcanzó una exactitud del 94%, lo que indica el buen comportamiento del modelo frente a la clasificación de los clientes con y sin riesgo de lavado de activos.

Se destaca la precisión del 94.67%, lo que demuestra el buen funcionamiento del modelo al momento de identificar correctamente los casos asociados a lavado de activos. Igualmente, se puede observar el buen desempeño del modelo frente a la clasificación de clientes que no presentan riesgo, tal como lo demuestra el índice de especificidad del 98.02%, el cual evidencia una alta capacidad para reconocer correctamente a los clientes normales.

Es importante mencionar que, aunque la exactitud general es elevada, aún existen errores de clasificación representados por los falsos positivos y falsos negativos. Esto se evidencia en la sensibilidad del 84.36% y en la predicción negativa del 93.76%, lo que indica que todavía hay margen de mejora en la detección total de los casos reales de lavado de activos.